<a href="https://colab.research.google.com/github/pdf1802/f1-race-replay/blob/feature%2Fdata-analysis/analysis/notebooks/driver_style_fingerprinting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3 — Driver Style Fingerprinting

**Goal:** Use telemetry data to extract each driver's style signature
and cluster them using unsupervised ML.

**Approach:** We use qualifying laps — no traffic, no strategy,
maximum effort from every driver. Pure style signal.

**Pipeline:** Load → Resample to distance grid → Engineer features
→ Aggregate per driver → Scale → PCA → K-Means → UMAP → Visualize

In [1]:
!pip install fastf1 umap-learn plotly --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.1 MB/s eta 0:00:00


In [2]:
import fastf1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.interpolate import interp1d
import umap
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

CACHE_DIR = '/content/drive/MyDrive/f1_cache'
fastf1.Cache.enable_cache(CACHE_DIR)
print("✅ Setup complete")

Mounted at /content/drive
✅ Setup complete


In [3]:
#We start with Bahrain 22024 Qualifying
# 'Q' = Qualifying session in FastF1
session = fastf1.get_session(2024, 'Bahrain', 'Q')
session.load()

#Check what drivers are available
print(f"Drivers in session: {list(session.drivers)}")
print(f"\nTotal laps loaded: {len(session.laps)}")

core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Bahrain Grand Prix - Qualifying [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
INFO:fastf1.fastf1.req:No cached data found for sess

Drivers in session: ['1', '16', '63', '55', '11', '14', '4', '81', '44', '27', '22', '18', '23', '3', '20', '77', '24', '2', '31', '10']

Total laps loaded: 267


In [4]:
# Always inspect raw data before writing any feature engineering
# Pick Verstappen's fastest qualifying lap
driver_code = 'VER'
lap = session.laps.pick_driver(driver_code).pick_fastest()
tel = lap.get_telemetry()

print(f"Telemetry columns: {tel.columns.tolist()}")
print(f"Number of datapoints: {len(tel)}")
print(f"Distance range: {tel['Distance'].min():.1f}m → {tel['Distance'].max():.1f}m")
print(f"\nFirst 5 rows:")
tel[['Distance', 'Speed', 'Throttle', 'Brake', 'DRS', 'nGear']].head()

Telemetry columns: ['Date', 'SessionTime', 'DriverAhead', 'DistanceToDriverAhead', 'Time', 'RPM', 'Speed', 'nGear', 'Throttle', 'Brake', 'DRS', 'Source', 'Distance', 'RelativeDistance', 'Status', 'X', 'Y', 'Z']
Number of datapoints: 702
Distance range: 0.1m → 5366.3m

First 5 rows:


,Distance,Speed,Throttle,Brake,DRS,nGear
2,0.051314,297.241668,100.0,False,12,7
3,3.288372,297.566667,100.0,False,12,7
4,7.615556,298.000000,100.0,False,12,8
5,18.231391,300.381251,100.0,False,12,8
6,20.993333,301.000000,100.0,False,12,8


In [5]:
def resample_to_distance_grid(tel, n_points=300):
    """
    Interpolate a telemetry DataFrame onto a fixed distance grid.

    Why: Every circuit has a different lap length → different number of
    telemetry points. We can't compare features across circuits unless
    all laps are the same 'shape'. 300 points is enough resolution to
    capture braking zones without noise.
    """
    # Define the common grid: 0 to 1 (relative distance)
    # We use RelativeDistance so it works for ANY circuit length
    grid = np.linspace(0, 1, n_points)

    resampled = {}
    channels = ['Speed', 'Throttle', 'Brake', 'DRS']

    dist = tel['RelativeDistance'].values

    for ch in channels:
        values = tel[ch].astype(float).values  # cast bool Brake → float
        f = interp1d(dist, values, bounds_error=False, fill_value='extrapolate')
        resampled[ch] = f(grid)

    return pd.DataFrame(resampled, index=grid)

# Test it on VER's lap
resampled = resample_to_distance_grid(tel)
print(f"Resampled shape: {resampled.shape}")  # Should be (300, 4)
print(resampled.head())

Resampled shape: (300, 4)
               Speed  Throttle  Brake   DRS
0.000000  297.236516     100.0    0.0  12.0
0.003344  300.326959     100.0    0.0  12.0
0.006689  301.636840     100.0    0.0  12.0
0.010033  303.974174     100.0    0.0  12.0
0.013378  307.340309     100.0    0.0  12.0


## Step 1 — Resample to Fixed Distance Grid

Raw telemetry varies by circuit: Bahrain = 702 points over 5,366m,
Spa = ~1,000 points over 7,004m. We can't compare features across
circuits unless every lap has the same shape.

`resample_to_distance_grid()` uses `scipy.interp1d` to interpolate
every lap onto 300 evenly-spaced points from 0% to 100% of the lap
(`RelativeDistance`). After this, point #150 always means
"halfway through the lap" regardless of circuit.

**Why 300 points?** Enough resolution to capture braking zones
(which last ~50–80m) without adding noise.

In [6]:
def extract_style_features(resampled_tel):
    """
    From a 300-point resampled telemetry DataFrame, extract 6 style features.
    Returns a dict — one row worth of data.

    Note: Brake is boolean in FastF1 → we measure application % not pressure.
    """
    throttle = resampled_tel['Throttle'].values  # 0-100
    brake    = resampled_tel['Brake'].values      # 0.0 or 1.0 after cast
    speed    = resampled_tel['Speed'].values
    drs      = resampled_tel['DRS'].values

    # 1. What % of lap is the brake pedal applied?
    brake_application_pct = brake.mean()

    # 2. What % of lap is full throttle (>98%)? — commitment on straights
    full_throttle_pct = (throttle > 98).mean()

    # 3. Coasting: neither throttle nor brake — the "float" through corners
    coasting_pct = ((throttle < 2) & (brake < 0.5)).mean()

    # 4. Throttle smoothness: std of lap-to-lap throttle changes
    # High std = choppy/aggressive; Low std = smooth/progressive
    throttle_smoothness = np.std(np.diff(throttle))

    # 5. Min speed ratio: lowest corner speed / top speed
    # High = carries more speed through corners (mechanical grip style)
    min_speed_ratio = speed.min() / speed.max()

    # 6. DRS usage: % of lap with DRS fully open (value == 12)
    drs_usage_pct = (drs == 12).mean()

    return {
        'brake_application_pct': brake_application_pct,
        'full_throttle_pct':     full_throttle_pct,
        'coasting_pct':          coasting_pct,
        'throttle_smoothness':   throttle_smoothness,
        'min_speed_ratio':       min_speed_ratio,
        'drs_usage_pct':         drs_usage_pct,
    }

# Test on VER
features = extract_style_features(resampled)
print("VER features — Bahrain Q 2024:")
for k, v in features.items():
    print(f"  {k}: {v:.4f}")

VER features — Bahrain Q 2024:
  brake_application_pct: 0.1420
  full_throttle_pct: 0.7133
  coasting_pct: 0.0067
  throttle_smoothness: 15.3372
  min_speed_ratio: 0.2218
  drs_usage_pct: 0.3033


## Step 2 — Extract Style Features

Turns the 300-point trace into 6 numbers that describe *how* the
driver drove — their style signature.

| Feature | Meaning |
|---|---|
| `brake_application_pct` | % of lap with brake pedal applied |
| `full_throttle_pct` | % of lap at full throttle (>98%) |
| `coasting_pct` | % of lap with neither throttle nor brake |
| `throttle_smoothness` | std of throttle changes — smooth vs aggressive |
| `min_speed_ratio` | min corner speed / top speed |
| `drs_usage_pct` | % of lap with DRS fully open |

**Note:** FastF1 `Brake` is boolean (on/off), not pressure.
So we measure *when* they brake, not *how hard*.

## Step 3 — Loop All Drivers

Runs Steps 1 and 2 for every driver in the session automatically.

`try/except` is essential — some drivers have incomplete telemetry
(crash in Q1, mechanical issues). Without it, one bad driver crashes
the entire loop and we lose all collected data.

Output: one row per driver, 6 features + Driver, Team, Circuit columns.

In [7]:
def get_session_features(session):
    """
    For a single qualifying session, extract style features for every driver
    that has clean telemetry on their fastest lap.
    Returns a DataFrame — one row per driver.
    """
    rows = []

    for driver in session.drivers:
        try:
            lap = session.laps.pick_driver(driver).pick_fastest()

            # Skip if no valid lap
            if lap.empty:
                continue

            tel = lap.get_telemetry()

            # Skip if telemetry too short (incomplete lap)
            if len(tel) < 100:
                continue

            resampled = resample_to_distance_grid(tel)
            feats = extract_style_features(resampled)

            feats['Driver']  = driver
            feats['Team']    = lap['Team']
            feats['Circuit'] = session.event['EventName']
            rows.append(feats)

        except Exception as e:
            print(f"  Skipping {driver}: {e}")
            continue

    return pd.DataFrame(rows)

# Run on Bahrain Q 2024
print("Extracting features for all drivers — Bahrain Q 2024...")
df_bahrain = get_session_features(session)
print(f"\nShape: {df_bahrain.shape}")
print(df_bahrain[['Driver', 'Team', 'brake_application_pct', 'coasting_pct', 'throttle_smoothness']].to_string())

Extracting features for all drivers — Bahrain Q 2024...

Shape: (20, 9)
   Driver             Team  brake_application_pct  coasting_pct  throttle_smoothness
0       1  Red Bull Racing               0.142007      0.006667            15.337183
1      16          Ferrari               0.123545      0.006667            13.223558
2      63         Mercedes               0.126045      0.020000            14.673870
3      55          Ferrari               0.132147      0.010000            14.240274
4      11  Red Bull Racing               0.134153      0.020000            16.416176
5      14     Aston Martin               0.122262      0.020000            16.052680
6       4          McLaren               0.124846      0.016667            13.797047
7      81          McLaren               0.131125      0.023333            14.626840
8      44         Mercedes               0.134983      0.000000            13.398846
9      27     Haas F1 Team               0.124055      0.016667            15.

## First Observations — Bahrain Q 2024

Even before clustering, patterns are visible in the raw features:

- **HAM** `coasting_pct = 0.000` — never coasts, always on throttle or brake
- **LEC** `throttle_smoothness = 13.22` — smoothest inputs of all top drivers  
- **PER** `throttle_smoothness = 16.42` — choppiest inputs, consistent with his tyre management issues
- **ALB/MAG** `brake_application_pct ~0.148` — midfield cars brake more to compensate for less downforce

These align with real F1 knowledge → good signal before we even cluster.

**Next:** Load 5 more circuits, aggregate per driver, then PCA → K-Means → UMAP.